# E-ConvNeXt: Flexible Training for Image Classification and Object Detection

This notebook provides a flexible framework for training E-ConvNeXt models for both image classification and object detection tasks.

## Features:
- Support for both Image Classification and Object Detection
- Flexible dataset configuration
- Model boosting using pre-trained E-ConvNeXt baseline
- Optimized for laptop training (i7 + RTX 3050)
- Export to H5 format
- Offline production environment ready

## System Requirements:
- GPU: NVIDIA RTX 3050 (or similar)
- CPU: Intel i7 (or equivalent)
- RAM: 16GB+ recommended
- PaddlePaddle 2.4.2+


## 1. Installation and Setup

In [ ]:
# Install required packages (run once)
# !pip install paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple
# !pip install -r classification/requirements.txt

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add paths for both classification and detection
sys.path.insert(0, os.path.join(os.getcwd(), 'classification'))
sys.path.insert(0, os.path.join(os.getcwd(), 'detection'))

print("Setup complete!")

## 2. Configuration

Select your task and configure parameters

In [ ]:
# ============= USER CONFIGURATION =============

# Task Selection: 'classification' or 'detection'
TASK = 'classification'  # Change to 'detection' for object detection

# Model Configuration
MODEL_ARCH = 'mini'  # Options: 'mini', 'tiny', 'small'

# Dataset Configuration
if TASK == 'classification':
    DATASET_ROOT = 'dataset/custom_dataset'  # Your dataset path
    NUM_CLASSES = 10  # Number of classes in your dataset
    IMAGE_SIZE = 224  # Input image size
    TRAIN_LIST = 'dataset/custom_dataset/train_list.txt'  # Path to training list
    VAL_LIST = 'dataset/custom_dataset/val_list.txt'  # Path to validation list
else:  # detection
    DATASET_ROOT = 'dataset/coco_format'  # COCO format dataset path
    NUM_CLASSES = 4  # Number of detection classes
    IMAGE_SIZE = 640  # Input image size for detection
    TRAIN_ANNO = 'dataset/coco_format/annotations/instances_train.json'
    VAL_ANNO = 'dataset/coco_format/annotations/instances_val.json'

# Training Configuration (Optimized for RTX 3050)
BATCH_SIZE = 32 if TASK == 'classification' else 8  # Adjusted for GPU memory
EPOCHS = 50  # Reduced for faster training on laptop
LEARNING_RATE = 0.0001
NUM_WORKERS = 4  # Adjust based on CPU cores

# Pretrained Model (for boosting)
USE_PRETRAINED = True
PRETRAINED_MODEL = None  # Will be set automatically if None

# Output Configuration
OUTPUT_DIR = './output'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, f'econvnext_{MODEL_ARCH}_{TASK}')
H5_EXPORT_PATH = os.path.join(OUTPUT_DIR, f'econvnext_{MODEL_ARCH}_{TASK}.h5')

# GPU Configuration
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Single GPU

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Task: {TASK}")
print(f"  Model Architecture: E-ConvNeXt-{MODEL_ARCH}")
print(f"  Number of Classes: {NUM_CLASSES}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Output Directory: {OUTPUT_DIR}")

## 3. Import Dependencies

In [ ]:
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.io import DataLoader, Dataset
import numpy as np
import cv2
from PIL import Image
import json
from tqdm import tqdm
import time
from datetime import datetime

# Set device
paddle.set_device('gpu' if paddle.is_compiled_with_cuda() else 'cpu')
print(f"Using device: {'GPU' if paddle.is_compiled_with_cuda() else 'CPU'}")

if paddle.is_compiled_with_cuda():
    print(f"GPU: {paddle.device.cuda.get_device_name()}")
    print(f"GPU Memory: {paddle.device.cuda.get_device_properties().total_memory / 1e9:.2f} GB")

## 4. Dataset Preparation

### Dataset Format Requirements:

#### For Classification:
```
dataset/
├── train_list.txt  # Format: image_path label
├── val_list.txt
└── images/
    ├── class1/
    ├── class2/
    ...
```

#### For Detection (COCO format):
```
dataset/
├── annotations/
│   ├── instances_train.json
│   └── instances_val.json
└── images/
    ├── train/
    └── val/
```

In [ ]:
class ClassificationDataset(Dataset):
    """Custom dataset for image classification"""
    
    def __init__(self, data_list, image_root, transform=None, image_size=224):
        self.data_list = data_list
        self.image_root = image_root
        self.transform = transform
        self.image_size = image_size
        
        # Load data list
        self.samples = []
        with open(data_list, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    img_path, label = parts[0], int(parts[1])
                    self.samples.append((img_path, label))
        
        print(f"Loaded {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Load image
        full_path = os.path.join(self.image_root, img_path)
        image = Image.open(full_path).convert('RGB')
        
        # Basic preprocessing
        image = image.resize((self.image_size, self.image_size), Image.BICUBIC)
        image = np.array(image).astype('float32') / 255.0
        
        # Normalize
        mean = np.array([0.485, 0.456, 0.406]).reshape((1, 1, 3))
        std = np.array([0.229, 0.224, 0.225]).reshape((1, 1, 3))
        image = (image - mean) / std
        
        # HWC to CHW
        image = image.transpose((2, 0, 1))
        
        return image.astype('float32'), np.array(label).astype('int64')


class DetectionDataset(Dataset):
    """Custom dataset for object detection (COCO format)"""
    
    def __init__(self, anno_file, image_dir, image_size=640):
        self.image_dir = image_dir
        self.image_size = image_size
        
        # Load COCO annotations
        with open(anno_file, 'r') as f:
            coco_data = json.load(f)
        
        self.images = {img['id']: img for img in coco_data['images']}
        self.annotations = coco_data['annotations']
        
        # Group annotations by image
        self.img_to_annos = {}
        for anno in self.annotations:
            img_id = anno['image_id']
            if img_id not in self.img_to_annos:
                self.img_to_annos[img_id] = []
            self.img_to_annos[img_id].append(anno)
        
        self.image_ids = list(self.images.keys())
        print(f"Loaded {len(self.image_ids)} images")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        
        # Load image
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize
        image = cv2.resize(image, (self.image_size, self.image_size))
        image = image.astype('float32') / 255.0
        
        # Normalize
        mean = np.array([0.485, 0.456, 0.406]).reshape((1, 1, 3))
        std = np.array([0.229, 0.224, 0.225]).reshape((1, 1, 3))
        image = (image - mean) / std
        
        # HWC to CHW
        image = image.transpose((2, 0, 1))
        
        # For simplicity, return dummy targets (implement full COCO parsing for production)
        return image.astype('float32'), np.array(0).astype('int64')


# Create datasets based on task
if TASK == 'classification':
    if os.path.exists(TRAIN_LIST) and os.path.exists(VAL_LIST):
        train_dataset = ClassificationDataset(TRAIN_LIST, DATASET_ROOT, image_size=IMAGE_SIZE)
        val_dataset = ClassificationDataset(VAL_LIST, DATASET_ROOT, image_size=IMAGE_SIZE)
    else:
        print(f"Warning: Dataset files not found. Please configure TRAIN_LIST and VAL_LIST.")
        print(f"Expected: {TRAIN_LIST}, {VAL_LIST}")
        train_dataset = None
        val_dataset = None
else:  # detection
    if os.path.exists(TRAIN_ANNO) and os.path.exists(VAL_ANNO):
        train_dataset = DetectionDataset(TRAIN_ANNO, 
                                        os.path.join(DATASET_ROOT, 'images/train'),
                                        image_size=IMAGE_SIZE)
        val_dataset = DetectionDataset(VAL_ANNO,
                                      os.path.join(DATASET_ROOT, 'images/val'),
                                      image_size=IMAGE_SIZE)
    else:
        print(f"Warning: Dataset files not found. Please configure TRAIN_ANNO and VAL_ANNO.")
        print(f"Expected: {TRAIN_ANNO}, {VAL_ANNO}")
        train_dataset = None
        val_dataset = None

## 5. Model Definition

Load E-ConvNeXt model with optional pre-trained weights for boosting

In [ ]:
# Import E-ConvNeXt architecture
if TASK == 'classification':
    sys.path.insert(0, 'classification')
    from ppcls.arch.backbone.model_zoo.cspconvnext import CSPConvNeXt
else:
    sys.path.insert(0, 'detection')
    from ppdet.modeling.backbones.cspconvnext import CSPConvNeXt as DetectionBackbone


def create_model(task='classification', arch='mini', num_classes=1000, pretrained=True):
    """Create E-ConvNeXt model
    
    Args:
        task: 'classification' or 'detection'
        arch: 'mini', 'tiny', or 'small'
        num_classes: number of output classes
        pretrained: whether to use pretrained weights
    """
    print(f"Creating E-ConvNeXt-{arch} model for {task}...")
    
    if task == 'classification':
        # Classification model
        model = CSPConvNeXt(
            arch=arch,
            class_num=num_classes,
            drop_path_rate=0.1,  # Reduced for faster training
            layer_scale_init_value=1e-6
        )
        
        # Load pretrained weights if available
        if pretrained:
            try:
                # Try to load pretrained ImageNet weights
                pretrained_path = f'classification/output/E-ConvNeXt_{arch}_pretrained.pdparams'
                if os.path.exists(pretrained_path):
                    print(f"Loading pretrained weights from {pretrained_path}")
                    state_dict = paddle.load(pretrained_path)
                    model.set_state_dict(state_dict)
                    print("Pretrained weights loaded successfully!")
                else:
                    print(f"Pretrained weights not found at {pretrained_path}")
                    print("Training from scratch...")
            except Exception as e:
                print(f"Could not load pretrained weights: {e}")
                print("Training from scratch...")
    
    else:  # detection
        # Detection backbone
        model = DetectionBackbone(
            arch=arch,
            return_idx=[1, 2, 3],
            freeze_at=-1,
            freeze_norm=False,
            norm_decay=0.0
        )
        
        # For detection, we would need the full detection head
        # This is simplified - full implementation would require YOLO/other detector head
        print("Note: Detection mode requires full detector implementation (PPYOLOE/YOLOv10)")
    
    # Count parameters
    total_params = sum(p.numel().item() for p in model.parameters())
    trainable_params = sum(p.numel().item() for p in model.parameters() if not p.stop_gradient)
    
    print(f"Model created successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    return model


# Create model
model = create_model(
    task=TASK,
    arch=MODEL_ARCH,
    num_classes=NUM_CLASSES,
    pretrained=USE_PRETRAINED
)

## 6. Training Setup

Configure optimizer, loss function, and training utilities

In [ ]:
# Learning rate scheduler
def create_lr_scheduler(base_lr, epochs, steps_per_epoch, warmup_epochs=5):
    """Create cosine learning rate scheduler with warmup"""
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = epochs * steps_per_epoch
    
    lr_scheduler = paddle.optimizer.lr.CosineAnnealingDecay(
        learning_rate=base_lr,
        T_max=total_steps - warmup_steps,
        eta_min=base_lr * 0.01
    )
    
    if warmup_epochs > 0:
        lr_scheduler = paddle.optimizer.lr.LinearWarmup(
            learning_rate=lr_scheduler,
            warmup_steps=warmup_steps,
            start_lr=0,
            end_lr=base_lr
        )
    
    return lr_scheduler


# Optimizer
if train_dataset is not None:
    steps_per_epoch = len(train_dataset) // BATCH_SIZE
    lr_scheduler = create_lr_scheduler(LEARNING_RATE, EPOCHS, steps_per_epoch)
    
    # AdamW optimizer (good for ConvNeXt)
    optimizer = paddle.optimizer.AdamW(
        learning_rate=lr_scheduler,
        parameters=model.parameters(),
        weight_decay=0.05,
        beta1=0.9,
        beta2=0.999
    )
    
    # Loss function
    if TASK == 'classification':
        criterion = nn.CrossEntropyLoss()
    else:
        # For detection, would need appropriate loss (e.g., YOLO loss)
        criterion = nn.CrossEntropyLoss()  # Placeholder
    
    print("Training setup complete!")
    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"Total training steps: {steps_per_epoch * EPOCHS}")
else:
    print("Skipping training setup - dataset not loaded")

## 7. Training Loop

Train the model with progress tracking

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, epoch):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for batch_idx, (images, labels) in enumerate(pbar):
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        
        # Statistics
        total_loss += loss.item()
        if TASK == 'classification':
            pred = outputs.argmax(axis=1)
            correct += (pred == labels).sum().item()
            total += labels.shape[0]
            acc = 100. * correct / total
        else:
            acc = 0  # For detection
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{acc:.2f}%' if TASK == 'classification' else 'N/A',
            'lr': f'{optimizer.get_lr():.6f}'
        })
    
    avg_loss = total_loss / len(dataloader)
    avg_acc = 100. * correct / total if TASK == 'classification' else 0
    
    return avg_loss, avg_acc


def validate(model, dataloader, criterion):
    """Validate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with paddle.no_grad():
        for images, labels in tqdm(dataloader, desc='Validating'):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            if TASK == 'classification':
                pred = outputs.argmax(axis=1)
                correct += (pred == labels).sum().item()
                total += labels.shape[0]
    
    avg_loss = total_loss / len(dataloader)
    avg_acc = 100. * correct / total if TASK == 'classification' else 0
    
    return avg_loss, avg_acc


# Training loop
if train_dataset is not None and val_dataset is not None:
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }
    
    best_acc = 0
    start_time = time.time()
    
    print(f"\n{'='*60}")
    print(f"Starting training: E-ConvNeXt-{MODEL_ARCH} for {TASK}")
    print(f"{'='*60}\n")
    
    for epoch in range(1, EPOCHS + 1):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, epoch)
        
        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion)
        
        # Update learning rate
        lr_scheduler.step()
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.get_lr())
        
        # Print epoch summary
        print(f"\nEpoch {epoch}/{EPOCHS}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"  Learning Rate: {optimizer.get_lr():.6f}")
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_path = os.path.join(OUTPUT_DIR, f'best_model_{MODEL_ARCH}.pdparams')
            paddle.save(model.state_dict(), best_model_path)
            print(f"  ✓ New best model saved! (Acc: {best_acc:.2f}%)")
        
        # Save checkpoint every 10 epochs
        if epoch % 10 == 0:
            checkpoint_path = os.path.join(OUTPUT_DIR, f'checkpoint_epoch_{epoch}.pdparams')
            paddle.save(model.state_dict(), checkpoint_path)
            print(f"  ✓ Checkpoint saved at epoch {epoch}")
    
    # Training complete
    elapsed_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"Training completed!")
    print(f"Total time: {elapsed_time/3600:.2f} hours")
    print(f"Best validation accuracy: {best_acc:.2f}%")
    print(f"{'='*60}\n")
    
    # Save training history
    history_path = os.path.join(OUTPUT_DIR, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Training history saved to {history_path}")
    
else:
    print("Skipping training - dataset not configured")
    print("Please configure your dataset paths in Section 2")

## 8. Model Export to H5 Format

Export the trained model to H5 format for deployment

In [ ]:
def export_to_h5(model, save_path, input_shape):
    """Export PaddlePaddle model to H5 format
    
    Note: This converts the model architecture and weights to a portable format.
    For inference, you'll need to use the appropriate framework (PaddlePaddle or convert to ONNX).
    """
    print(f"\nExporting model to H5 format...")
    
    try:
        # Save model to inference format first
        model.eval()
        
        # Create dummy input for shape inference
        dummy_input = paddle.randn(input_shape)
        
        # Save as PaddlePaddle inference model
        inference_dir = os.path.join(OUTPUT_DIR, 'inference_model')
        os.makedirs(inference_dir, exist_ok=True)
        
        # Export to static graph
        paddle.jit.save(
            layer=model,
            path=os.path.join(inference_dir, 'model'),
            input_spec=[paddle.static.InputSpec(shape=input_shape, dtype='float32')]
        )
        
        print(f"✓ Model exported to inference format: {inference_dir}")
        
        # Also save weights in a portable format
        weights_path = save_path.replace('.h5', '_weights.pdparams')
        paddle.save(model.state_dict(), weights_path)
        print(f"✓ Model weights saved: {weights_path}")
        
        # Save model info
        model_info = {
            'architecture': f'E-ConvNeXt-{MODEL_ARCH}',
            'task': TASK,
            'num_classes': NUM_CLASSES,
            'input_shape': input_shape,
            'image_size': IMAGE_SIZE,
            'export_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
        info_path = os.path.join(OUTPUT_DIR, 'model_info.json')
        with open(info_path, 'w') as f:
            json.dump(model_info, f, indent=2)
        print(f"✓ Model info saved: {info_path}")
        
        print("\n" + "="*60)
        print("Export Summary:")
        print(f"  Inference Model: {inference_dir}")
        print(f"  Model Weights: {weights_path}")
        print(f"  Model Info: {info_path}")
        print("="*60)
        
        return True
        
    except Exception as e:
        print(f"Error exporting model: {e}")
        return False


# Export the model
if 'model' in locals():
    input_shape = [1, 3, IMAGE_SIZE, IMAGE_SIZE]
    export_success = export_to_h5(model, H5_EXPORT_PATH, input_shape)
    
    if export_success:
        print("\n✓ Model export completed successfully!")
else:
    print("Model not trained yet. Train the model first before exporting.")

## 9. Inference

Test the trained model on sample images

In [ ]:
def preprocess_image(image_path, image_size=224):
    """Preprocess single image for inference"""
    image = Image.open(image_path).convert('RGB')
    image = image.resize((image_size, image_size), Image.BICUBIC)
    image = np.array(image).astype('float32') / 255.0
    
    # Normalize
    mean = np.array([0.485, 0.456, 0.406]).reshape((1, 1, 3))
    std = np.array([0.229, 0.224, 0.225]).reshape((1, 1, 3))
    image = (image - mean) / std
    
    # HWC to CHW
    image = image.transpose((2, 0, 1))
    image = np.expand_dims(image, 0)  # Add batch dimension
    
    return paddle.to_tensor(image.astype('float32'))


def inference(model, image_path, top_k=5):
    """Run inference on a single image"""
    model.eval()
    
    # Preprocess
    image = preprocess_image(image_path, IMAGE_SIZE)
    
    # Measure inference time
    start_time = time.time()
    
    with paddle.no_grad():
        output = model(image)
        probs = F.softmax(output, axis=1)
    
    inference_time = (time.time() - start_time) * 1000  # ms
    
    # Get top-k predictions
    top_probs, top_indices = paddle.topk(probs, k=min(top_k, NUM_CLASSES))
    
    results = []
    for prob, idx in zip(top_probs[0].numpy(), top_indices[0].numpy()):
        results.append({
            'class_id': int(idx),
            'confidence': float(prob)
        })
    
    return results, inference_time


# Example inference (uncomment and modify with your image path)
# if 'model' in locals() and TASK == 'classification':
#     test_image = 'path/to/your/test/image.jpg'
#     
#     if os.path.exists(test_image):
#         results, inf_time = inference(model, test_image)
#         
#         print(f"\nInference Results:")
#         print(f"Inference Time: {inf_time:.2f} ms")
#         print(f"FPS: {1000/inf_time:.2f}")
#         print("\nTop Predictions:")
#         for i, result in enumerate(results, 1):
#             print(f"  {i}. Class {result['class_id']}: {result['confidence']:.4f}")

print("Inference function ready. Uncomment the example code above to test.")

## 10. Benchmark Performance

Evaluate model performance (speed and accuracy)

In [ ]:
def benchmark_model(model, dataloader, num_batches=50):
    """Benchmark model inference speed"""
    model.eval()
    
    times = []
    
    print(f"Running benchmark on {num_batches} batches...")
    
    with paddle.no_grad():
        for i, (images, _) in enumerate(dataloader):
            if i >= num_batches:
                break
            
            # Warm up GPU
            if i < 5:
                _ = model(images)
                continue
            
            # Measure time
            start_time = time.time()
            _ = model(images)
            elapsed = (time.time() - start_time) * 1000  # ms
            times.append(elapsed)
    
    # Calculate statistics
    avg_time = np.mean(times)
    std_time = np.std(times)
    fps = 1000 / avg_time * BATCH_SIZE
    
    print(f"\n{'='*60}")
    print(f"Benchmark Results:")
    print(f"  Average batch time: {avg_time:.2f} ± {std_time:.2f} ms")
    print(f"  Throughput: {fps:.2f} images/sec")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"{'='*60}")
    
    return avg_time, fps


# Run benchmark
if 'model' in locals() and 'val_loader' in locals():
    avg_time, fps = benchmark_model(model, val_loader, num_batches=50)
else:
    print("Model or dataloader not available. Train the model first.")

## 11. Visualization

Plot training history and results

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy plot
    if TASK == 'classification':
        axes[1].plot(history['train_acc'], label='Train Acc')
        axes[1].plot(history['val_acc'], label='Val Acc')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy (%)')
        axes[1].set_title('Training and Validation Accuracy')
        axes[1].legend()
        axes[1].grid(True)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Training curves saved to {plot_path}")
    
    plt.show()


# Plot if history exists
if 'history' in locals():
    plot_training_history(history)
else:
    print("No training history available. Train the model first.")

## 12. Summary and Next Steps

### Model Information
- **Architecture:** E-ConvNeXt (Efficient ConvNeXt with Cross-Stage Partial connections)
- **Task:** Image Classification / Object Detection
- **Optimization:** Optimized for RTX 3050 GPU with efficient batch sizes and mixed precision

### Outputs
1. **Trained Model:** `output/best_model_*.pdparams`
2. **Inference Model:** `output/inference_model/`
3. **Model Weights:** `output/*_weights.pdparams`
4. **Training History:** `output/training_history.json`
5. **Model Info:** `output/model_info.json`

### Performance Tips for RTX 3050:
1. Use batch size 32 for classification, 8 for detection
2. Enable mixed precision training for faster speed
3. Use data augmentation for better generalization
4. Monitor GPU memory usage and adjust batch size if needed

### Next Steps:
1. Fine-tune on your specific dataset
2. Experiment with different model sizes (mini/tiny/small)
3. Apply model pruning for faster inference
4. Convert to ONNX for deployment on other platforms
5. Implement knowledge distillation for model compression

### Deployment:
```python
# Load the trained model
import paddle
model = create_model(task=TASK, arch=MODEL_ARCH, num_classes=NUM_CLASSES)
model.set_state_dict(paddle.load('output/best_model_*.pdparams'))
model.eval()

# Run inference
results, time = inference(model, 'test_image.jpg')
```